# Sri Lankan Airlines Policy Assistant


In [ ]:
import shutil
from pathlib import Path

!pip install -q gdown
import gdown

DATASET_FOLDER_URL = "https://drive.google.com/drive/folders/1vyyhSn4leDS70P7dyifrak2Vn0nn-K34?usp=drive_link"
PROJECT_DIR = Path("/content/SLAL_chatbot")
LOCAL_DATASET_PATH = PROJECT_DIR / "dataset"

PROJECT_DIR.mkdir(parents=True, exist_ok=True)

if LOCAL_DATASET_PATH.exists():
    shutil.rmtree(LOCAL_DATASET_PATH)

LOCAL_DATASET_PATH.mkdir(parents=True, exist_ok=True)

gdown.download_folder(
    url=DATASET_FOLDER_URL,
    output=str(LOCAL_DATASET_PATH),
    quiet=False,
    use_cookies=False,
)

txt_files = sorted(LOCAL_DATASET_PATH.glob("*.txt"))
print(f"Downloaded {len(txt_files)} dataset files to {LOCAL_DATASET_PATH}")
for path in txt_files[:5]:
    print(" -", path.name)
if len(txt_files) > 5:
    print(f" ... and {len(txt_files) - 5} more")


In [ ]:
import os
from pathlib import Path

PROJECT_DIR = Path("/content/SLAL_chatbot")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(PROJECT_DIR)

CODE_FILES = {'build_index.py': 'import os\nimport re\nfrom langchain_community.document_loaders import TextLoader\nfrom langchain_core.documents import Document\nfrom langchain_community.vectorstores import FAISS\nfrom langchain_huggingface import HuggingFaceEmbeddings\n\nDATASET_PATH = "dataset"\nVECTOR_STORE_PATH = "vector_store"\nEMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"\n\nSECTION_PATTERN = re.compile(\n    r"={20,}\\n(?P<section_heading>.*?)\\n={20,}\\n\\n(?P<section_body>.*?)(?=\\n={20,}\\n.*?\\n={20,}\\n\\n|\\n={20,}\\nEND OF DOCUMENT\\n={20,}\\n?$)",\n    re.DOTALL,\n)\n\n\ndef extract_metadata_and_sections(doc):\n    text = doc.page_content\n    source = doc.metadata.get("source", "Unknown source")\n\n    title_match = re.search(r"Title:\\s*(.*?)\\n", text)\n    description_match = re.search(r"Description:\\s*(.*?)\\n\\n={20,}", text, re.DOTALL)\n\n    title = title_match.group(1).strip() if title_match else "Unknown title"\n    description = description_match.group(1).strip() if description_match else ""\n\n    sections = []\n    for match in SECTION_PATTERN.finditer(text):\n        section_heading = match.group("section_heading").strip()\n        section_body = match.group("section_body").strip()\n        section_id_match = re.match(r"^(\\d+(?:\\.\\d+)*)", section_heading)\n        section_id = section_id_match.group(1) if section_id_match else "unknown"\n\n        cleaned_content = (\n            f"Title: {title}\\n\\n"\n            f"Section: {section_heading}\\n\\n"\n            f"{section_body}"\n        )\n\n        sections.append(\n            Document(\n                page_content=cleaned_content,\n                metadata={\n                    "source": source,\n                    "title": title,\n                    "description": description,\n                    "section_id": section_id,\n                    "section_heading": section_heading,\n                },\n            )\n        )\n\n    return sections\n\n\ndef load_section_chunks():\n    documents = []\n    section_chunks = []\n\n    for file in sorted(os.listdir(DATASET_PATH)):\n        if not file.endswith(".txt"):\n            continue\n\n        file_path = os.path.join(DATASET_PATH, file)\n        docs = TextLoader(file_path, encoding="utf-8").load()\n        documents.extend(docs)\n\n        for doc in docs:\n            section_chunks.extend(extract_metadata_and_sections(doc))\n\n    return documents, section_chunks\n\n\ndef main():\n    documents, chunks = load_section_chunks()\n\n    # Use section-aware policy chunks instead of generic fixed-size splits.\n    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)\n    vectorstore = FAISS.from_documents(chunks, embeddings)\n\n    os.makedirs(VECTOR_STORE_PATH, exist_ok=True)\n    vectorstore.save_local(VECTOR_STORE_PATH)\n\n    print(f"Loaded {len(documents)} documents.")\n    print(f"Created {len(chunks)} section-aware chunks.")\n    print(f"Saved FAISS index to: {VECTOR_STORE_PATH}")\n\n\nif __name__ == "__main__":\n    main()', 'chatbot.py': 'from langchain_community.vectorstores import FAISS\nfrom langchain_core.prompts import PromptTemplate\nfrom langchain_huggingface import HuggingFaceEmbeddings\nfrom transformers import AutoModelForSeq2SeqLM, AutoTokenizer\nimport re\n\nVECTOR_STORE_PATH = "vector_store"\nEMBEDDING_MODEL = "sentence-transformers/all-MiniLM-L6-v2"\nLLM_MODEL = "google/flan-t5-base"\nTOP_K = 2\nSEMANTIC_K = 12\nFALLBACK_RESPONSE = "The information is unavailable in the Sri Lankan Airlines knowledge base."\nGUARDRAIL_RESPONSE = "This chatbot answers only Sri Lankan Airlines policy questions."\nTOXICITY_RESPONSE = "Please ask your question in respectful language."\nUNKNOWN_SOURCE = "Unknown source"\nMIN_RERANK_SCORE = 4\nMIN_SEMANTIC_SIMILARITY = 0.38\nSECTION_REF_PATTERN = re.compile(\n    r"(?:^|(?<=\\s))\\d{1,2}(?:\\.\\d{1,2}){1,3}\\s+(?=[A-Za-z])",\n    re.MULTILINE,\n)\nINJECTION_PATTERNS = (\n    re.compile(r"\\bignore\\b.{0,30}\\b(previous|above|earlier|prior)\\b.{0,20}\\b(instruction|prompt|message)s?\\b", re.IGNORECASE),\n    re.compile(r"\\bforget\\b.{0,30}\\b(previous|above|earlier|prior)\\b.{0,20}\\b(instruction|prompt|message)s?\\b", re.IGNORECASE),\n    re.compile(r"\\b(act|behave|pretend)\\b.{0,20}\\bas\\b", re.IGNORECASE),\n    re.compile(r"\\b(system prompt|developer message|hidden prompt|internal instruction)s?\\b", re.IGNORECASE),\n    re.compile(r"\\b(reveal|show|tell|print|expose)\\b.{0,20}\\b(system prompt|developer message|instructions?)\\b", re.IGNORECASE),\n)\nTOXIC_PATTERNS = (\n    re.compile(r"\\b(fuck|fucking|shit|bitch|bastard|asshole|moron|idiot|stupid)\\b", re.IGNORECASE),\n    re.compile(r"\\b(hate you|kill yourself|shut up|piece of shit)\\b", re.IGNORECASE),\n)\n\nPROMPT_TEMPLATE = """You are a Sri Lankan Airlines policy assistant.\nAnswer the question using only the policy text below.\nWrite clear sentences in plain English.\nIf the answer covers multiple classes, allowances, or policy sections, put each on its own line.\nDo not repeat headings, labels, or section titles.\nIf the policy text does not answer the question, reply exactly:\nI couldn\'t find this information in the knowledge base.\n\nPolicy text:\n{evidence}\n\nQuestion:\n{question}\n\nAnswer:\n"""\n\nLABEL_PATTERN = re.compile(\n    r"\\b(?:Title|Section|Excerpt|Source|Description)\\s*:\\s*",\n    re.IGNORECASE,\n)\nANSWER_BREAK_PATTERN = re.compile(\n    r"(?<!^)\\s+(?=(?:"\n    r"Business Class|"\n    r"Economy Class|"\n    r"Premium Economy(?: Class)?|"\n    r"First Class|"\n    r"Infant(?:\\s*\\([^)]*\\))?|"\n    r"\\d{1,2}(?:\\.\\d{1,2})+\\s+"\n    r"))",\n    re.IGNORECASE,\n)\n\n\ndef load_vector_store():\n    embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)\n    return FAISS.load_local(\n        VECTOR_STORE_PATH,\n        embeddings,\n        allow_dangerous_deserialization=True,\n    )\n\n\ndef load_llm():\n    tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL)\n    model = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL)\n    return tokenizer, model\n\n\ndef is_blocked_prompt(question):\n    normalized = re.sub(r"\\s+", " ", question.strip())\n    return any(pattern.search(normalized) for pattern in INJECTION_PATTERNS)\n\n\ndef is_toxic_input(question):\n    normalized = re.sub(r"\\s+", " ", question.strip())\n    return any(pattern.search(normalized) for pattern in TOXIC_PATTERNS)\n\n\ndef extract_keywords(question):\n    words = re.findall(r"\\b[a-zA-Z]{3,}\\b", question.lower())\n    stop_words = {\n        "what", "when", "where", "which", "who", "whom", "whose", "why", "how",\n        "does", "do", "did", "can", "could", "should", "would", "will",\n        "the", "and", "for", "are", "with", "from", "that", "this",\n        "have", "has", "had", "during", "into", "your", "their", "there",\n        "Sri Lankan", "airlines", "please", "tell", "about", "policy", "offer",\n        "passenger", "passengers",\n    }\n    return [word for word in words if word not in stop_words]\n\n\ndef extract_phrases(question):\n    lowered = question.lower()\n    phrases = []\n    for match in re.finditer(r"[a-z]+(?:-[a-z]+)+", lowered):\n        phrases.append(match.group(0))\n    keywords = extract_keywords(question)\n    phrases.extend(\n        f"{keywords[i]} {keywords[i + 1]}"\n        for i in range(len(keywords) - 1)\n    )\n    return phrases\n\n\ndef normalize_word(word):\n    normalized = word.lower()\n    if normalized.endswith("ies") and len(normalized) > 4:\n        normalized = normalized[:-3] + "y"\n    elif normalized.endswith("ing") and len(normalized) > 5:\n        normalized = normalized[:-3]\n    elif normalized.endswith("ed") and len(normalized) > 4:\n        normalized = normalized[:-2]\n    elif normalized.endswith("s") and len(normalized) > 4:\n        normalized = normalized[:-1]\n    if len(normalized) > 2 and normalized[-1] == normalized[-2]:\n        normalized = normalized[:-1]\n    return normalized\n\n\ndef tokenize_text(text):\n    return [normalize_word(token) for token in re.findall(r"\\b[a-zA-Z]{3,}\\b", text.lower())]\n\n\ndef policy_body(text):\n    # Strip metadata labels so scoring focuses on the policy content itself.\n    lines = []\n    for line in text.splitlines():\n        stripped = line.strip()\n        if not stripped:\n            continue\n        if stripped.startswith(("Title:", "Source:", "Description:", "Section:")):\n            continue\n        if "====" in stripped or stripped == "END OF DOCUMENT":\n            continue\n        lines.append(stripped)\n    return " ".join(lines)\n\n\ndef score_document(question, doc):\n    # Combine lexical overlap with a few light reranking boosts.\n    keywords = {normalize_word(w) for w in extract_keywords(question)}\n    phrases = extract_phrases(question)\n    body = policy_body(doc.page_content)\n    body_lower = body.lower()\n    body_tokens = set(tokenize_text(body))\n    heading = doc.metadata.get("section_heading", "").lower()\n\n    score = sum(len(k) for k in keywords if k in body_tokens)\n    score += 5 * sum(1 for p in phrases if p in body_lower)\n\n    title = doc.metadata.get("title", "")\n    meta_tokens = set(tokenize_text(f"{title} {heading}"))\n    score += 6 * len(keywords & meta_tokens)\n\n    question_lower = question.lower()\n    if "not allowed" in question_lower or "not permitted" in question_lower:\n        if any(w in heading for w in ("restricted", "prohibited", "unacceptable")):\n            score += 20\n        if any(w in body_lower for w in ("prohibited", "must not", "unacceptable", "not be permitted")):\n            score += 12\n        if "carry-on" in question_lower and "allowance" in heading:\n            score -= 15\n\n    if "policy" in question_lower and "general" in heading:\n        score += 12\n\n    if "check-in" in question_lower or "boarding" in question_lower:\n        if "check-in" in heading or "boarding" in heading:\n            score += 15\n\n    return score\n\n\ndef select_documents(question, vector_store):\n    candidates = vector_store.similarity_search(question, k=SEMANTIC_K)\n    if not candidates:\n        return [], 0\n\n    # Deduplicate sections before reranking so one section is not repeated.\n    seen = set()\n    unique_candidates = []\n    for doc in candidates:\n        key = (doc.metadata.get("source"), doc.metadata.get("section_id"))\n        if key not in seen:\n            seen.add(key)\n            unique_candidates.append(doc)\n\n    ranked = sorted(\n        ((score_document(question, doc), doc) for doc in unique_candidates),\n        key=lambda item: item[0],\n        reverse=True,\n    )\n\n    if not ranked or ranked[0][0] < MIN_RERANK_SCORE:\n        return [], ranked[0][0] if ranked else 0\n\n    best_score = ranked[0][0]\n    selected = [\n        doc for score, doc in ranked\n        if score >= max(best_score - 2, MIN_RERANK_SCORE)\n    ][:TOP_K]\n    return selected, best_score\n\n\ndef split_sentences(text):\n    parts = re.split(\n        r"(?<=[.!?])\\s+|(?<=;)\\s+|(?<=:)\\s+(?=\\([a-z]\\))|(?<=\\))\\s+(?=\\([a-z]\\))",\n        text,\n    )\n    return [p.strip() for p in parts if p.strip()]\n\n\ndef best_excerpt(doc, question, max_sentences=3):\n    body = policy_body(doc.page_content)\n    keywords = {normalize_word(w) for w in extract_keywords(question)}\n    phrases = extract_phrases(question)\n    sentences = split_sentences(body)\n\n    scored = []\n    for idx, sentence in enumerate(sentences):\n        tokens = set(tokenize_text(sentence))\n        s = sum(len(k) for k in keywords if k in tokens)\n        s += 5 * sum(1 for p in phrases if p in sentence.lower())\n        if s > 0:\n            scored.append((s, idx, sentence))\n            if sentence.endswith(":") and idx + 1 < len(sentences):\n                scored.append((s + 1, idx + 1, sentences[idx + 1]))\n\n    if not scored:\n        if keywords and not any(\n            keyword_matches(kw, body.lower(), set(tokenize_text(body)))\n            for kw in keywords\n        ):\n            return ""\n        return body[:600]\n\n    scored.sort(key=lambda item: (-item[0], item[1]))\n    top = scored[:max_sentences]\n    ordered = [sent for _, _, sent in sorted(top, key=lambda item: item[1])]\n    return " ".join(ordered)\n\n\ndef build_evidence(documents, question):\n    # Evidence is passed to the LLM as short grounded excerpts.\n    excerpts = []\n    for i, doc in enumerate(documents, start=1):\n        excerpt = best_excerpt(doc, question)\n        excerpts.append(f"[{i}] {excerpt}")\n    return "\\n\\n".join(excerpts)\n\n\ndef clean_final_answer(answer):\n    cleaned = answer.strip()\n    cleaned = LABEL_PATTERN.sub("", cleaned)\n    cleaned = re.sub(r"\\[\\d+\\]\\s*", "", cleaned)\n    cleaned = re.sub(r"^\\d{1,2}(?:\\.\\d{1,2}){1,3}\\s+(?=[A-Za-z])", "", cleaned)\n    cleaned = re.sub(r"[ \\t]+", " ", cleaned)\n    cleaned = re.sub(r"\\s*\\n\\s*", "\\n", cleaned)\n    return cleaned.strip()\n\n\ndef format_answer_display(answer):\n    if not answer or answer in {FALLBACK_RESPONSE, GUARDRAIL_RESPONSE, TOXICITY_RESPONSE}:\n        return answer\n\n    formatted = ANSWER_BREAK_PATTERN.sub("\\n", answer.strip())\n    lines = []\n    for line in formatted.splitlines():\n        line = line.strip()\n        if not line:\n            continue\n        line = SECTION_REF_PATTERN.sub("", line).strip()\n        if line:\n            lines.append(line)\n    return "\\n\\n".join(lines)\n\n\ndef capitalize_sentence(text):\n    if not text:\n        return text\n    return text[0].upper() + text[1:]\n\n\ndef evidence_to_answer(evidence):\n    text = LABEL_PATTERN.sub("", evidence)\n    text = re.sub(r"\\[\\d+\\]\\s*", "", text)\n    text = SECTION_REF_PATTERN.sub("", text)\n    parts = [p.strip() for p in re.split(r"\\n\\n+", text) if p.strip()]\n    if not parts:\n        return FALLBACK_RESPONSE\n    # Keep each retrieved excerpt on its own paragraph for readability.\n    answer = "\\n\\n".join(parts[:2])\n    answer = capitalize_sentence(answer)\n    if answer and answer[-1] not in ".!?":\n        answer += "."\n    return answer\n\n\ndef is_unusable_answer(answer):\n    cleaned = clean_final_answer(answer)\n    if not cleaned or cleaned == FALLBACK_RESPONSE:\n        return not cleaned\n    if len(cleaned.split()) < 8:\n        return True\n    if cleaned.endswith(":"):\n        return True\n    if LABEL_PATTERN.search(cleaned):\n        return True\n    if re.search(r"\\b(?:INVOLUNTARY|VOLUNTARY)\\s+REFUNDS\\b", cleaned):\n        return True\n    if cleaned.lower().startswith(("inform ", "obtain ", "acceptance of")):\n        return True\n    return False\n\n\ndef cosine_similarity(vec_a, vec_b):\n    dot = sum(a * b for a, b in zip(vec_a, vec_b))\n    norm_a = sum(a * a for a in vec_a) ** 0.5\n    norm_b = sum(b * b for b in vec_b) ** 0.5\n    if norm_a == 0 or norm_b == 0:\n        return 0.0\n    return dot / (norm_a * norm_b)\n\n\ndef semantic_similarity(question, evidence, vector_store):\n    embeddings = vector_store.embedding_function\n    question_vec = embeddings.embed_query(question)\n    evidence_vec = embeddings.embed_query(evidence)\n    return cosine_similarity(question_vec, evidence_vec)\n\n\ndef keyword_matches(keyword, evidence_lower, evidence_tokens):\n    normalized = normalize_word(keyword)\n    return normalized in evidence_tokens or normalized in evidence_lower\n\n\ndef evidence_supports_question(question, evidence, vector_store):\n    # Gate answers so unrelated retrievals fall back instead of hallucinating.\n    if semantic_similarity(question, evidence, vector_store) < MIN_SEMANTIC_SIMILARITY:\n        return False\n    return evidence_answers_question(question, evidence)\n\n\ndef evidence_answers_question(question, evidence):\n    keywords = extract_keywords(question)\n    if not keywords:\n        return False\n\n    evidence_lower = evidence.lower()\n    evidence_tokens = set(tokenize_text(evidence))\n    matches = [\n        kw for kw in keywords\n        if keyword_matches(kw, evidence_lower, evidence_tokens)\n    ]\n\n    if not matches:\n        return False\n    if len(keywords) == 1:\n        return True\n    if len(matches) >= 2:\n        return True\n    return len(matches) / len(keywords) >= 0.5\n\n\ndef generate_answer(question, evidence, llm, max_new_tokens=160):\n    prompt = PromptTemplate.from_template(PROMPT_TEMPLATE)\n    formatted = prompt.format(evidence=evidence, question=question)\n    tokenizer, model = llm\n    inputs = tokenizer(formatted, return_tensors="pt", truncation=True, max_length=1024)\n    outputs = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)\n    return clean_final_answer(tokenizer.decode(outputs[0], skip_special_tokens=True))\n\n\ndef answer_question(question, vector_store, llm):\n    if is_toxic_input(question):\n        return TOXICITY_RESPONSE, []\n\n    # Block basic prompt-injection attempts before retrieval or generation.\n    if is_blocked_prompt(question):\n        return GUARDRAIL_RESPONSE, []\n\n    docs, best_score = select_documents(question, vector_store)\n    if not docs:\n        return FALLBACK_RESPONSE, []\n\n    evidence = build_evidence(docs, question)\n    if not evidence.strip():\n        return FALLBACK_RESPONSE, docs\n\n    if not evidence_supports_question(question, evidence, vector_store):\n        return FALLBACK_RESPONSE, []\n\n    response = generate_answer(question, evidence, llm)\n    if is_unusable_answer(response):\n        response = generate_answer(question, evidence, llm, max_new_tokens=220)\n\n    if is_unusable_answer(response):\n        response = evidence_to_answer(evidence)\n\n    response = clean_final_answer(response)\n    response = capitalize_sentence(response)\n    if response and response[-1] not in ".!?":\n        response += "."\n\n    if (\n        response == FALLBACK_RESPONSE\n        and best_score >= MIN_RERANK_SCORE\n        and evidence_supports_question(question, evidence, vector_store)\n    ):\n        response = evidence_to_answer(evidence)\n\n    response = format_answer_display(response)\n    return response, docs\n\n\ndef main():\n    vector_store = load_vector_store()\n    llm = load_llm()\n    print("Chatbot is ready. Type \'exit\' to quit.\\n")\n\n    while True:\n        question = input("Ask a question: ").strip()\n\n        if question.lower() in {"exit", "quit"}:\n            print("Goodbye!")\n            break\n\n        if not question:\n            print("Please enter a question.\\n")\n            continue\n\n        answer, retrieved_docs = answer_question(question, vector_store, llm)\n\n        print("\\nAnswer:")\n        print(answer)\n        print("\\nRetrieved sources:")\n        seen = set()\n        for doc in retrieved_docs:\n            source = doc.metadata.get("source", UNKNOWN_SOURCE)\n            heading = doc.metadata.get("section_heading", "")\n            label = f"{source} ({heading})" if heading else source\n            if label not in seen:\n                print(f"- {label}")\n                seen.add(label)\n        print()\n\n\nif __name__ == "__main__":\n    main()\n', 'app.py': 'import gradio as gr\n\nfrom chatbot import answer_question, load_llm, load_vector_store\n\nVECTOR_STORE = load_vector_store()\nLLM = load_llm()\n\n\ndef format_sources(documents):\n    if not documents:\n        return "**Retrieved Sources**\\n\\nNo sources retrieved."\n\n    seen_sources = {}\n    for doc in documents:\n        source = doc.metadata.get("source", "Unknown source")\n        filename = source.rsplit("/", 1)[-1].rsplit("\\\\", 1)[-1]\n        if filename in seen_sources:\n            continue\n\n        section = (\n            doc.metadata.get("title", "").strip()\n            or doc.metadata.get("section_heading", "").strip()\n        )\n        if section == "Unknown title":\n            section = ""\n\n        seen_sources[filename] = section\n\n    lines = ["**Retrieved Sources**", ""]\n    for filename, section in seen_sources.items():\n        if section:\n            lines.append(f"- {filename} - Section: {section}")\n        else:\n            lines.append(f"- {filename}")\n\n    return "\\n".join(lines)\n\n\ndef respond(question):\n    question = question.strip()\n    if not question:\n        return "Please enter a question.", format_sources([])\n\n    answer, retrieved_docs = answer_question(question, VECTOR_STORE, LLM)\n    return answer, format_sources(retrieved_docs)\n\n\nwith gr.Blocks(title="Sri Lankan Airlines Policy Assistant") as demo:\n    gr.Markdown(\n        """\n        # Sri Lankan Airlines Policy Assistant\n        Ask questions about Sri Lankan Airlines Conditions of Carriage.\n\n        This chatbot retrieves relevant policy text from the knowledge base and\n        answers only from that retrieved information.\n        """\n    )\n\n    question_input = gr.Textbox(\n        label="Your Question",\n        placeholder="Example: What is the refund policy?",\n        lines=2,\n    )\n\n    ask_button = gr.Button("Ask")\n\n    answer_output = gr.Textbox(\n        label="Chatbot Answer",\n        lines=5,\n        interactive=False,\n    )\n\n    sources_output = gr.Markdown(label="Retrieved Sources")\n\n    # Support both button click and Enter key submission.\n    ask_button.click(fn=respond, inputs=question_input, outputs=[answer_output, sources_output])\n    question_input.submit(fn=respond, inputs=question_input, outputs=[answer_output, sources_output])\n\n\nif __name__ == "__main__":\n    demo.launch()\n'}

for filename, content in CODE_FILES.items():
    (PROJECT_DIR / filename).write_text(content, encoding="utf-8")

print(f"Project directory: {PROJECT_DIR}")
print("Created:", sorted(CODE_FILES.keys()))


In [ ]:
!pip install -q langchain langchain-community langchain-huggingface faiss-cpu sentence-transformers transformers torch gradio


In [ ]:
!python build_index.py


In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, "/content/SLAL_chatbot")

from app import demo

demo.launch(share=True)
